# Ellipsoid mesh test — pykarambola vs. analytical reference

Creates smooth parametric ellipsoid meshes at three resolutions and saves them as `.off` files.
Runs pykarambola directly on those meshes (no voxelization) and compares measured
Minkowski scalars and tensor β values against exact / numerically integrated analytical references.

**Purpose**: check whether the discrepancy between the fine-voxel reference and the analytical
value seen in `orientation_bias_archive.ipynb` is an artefact of the marching-cubes
discretisation, or whether it also appears when pykarambola is fed a smooth mesh directly.
The saved `.off` files can also be fed to the C++ karambola executable for a direct comparison.

**Shape**: triaxial ellipsoid, semi-axes A=10, B=6, C=4.
Mesh is a UV triangulation of the parametric surface `x(u,v) = (A cos u sin v, B sin u sin v, C cos v)`.
Three mesh resolutions are used (coarse / medium / fine).

In [13]:
import os
import numpy as np
import platform

import pykarambola as pk

print(f'pykarambola : {pk.__version__}')
print(f'Python      : {platform.python_version()}')
print(f'OS          : {platform.system()} {platform.release()}')

pykarambola : 0.5.0
Python      : 3.11.13
OS          : Darwin 24.6.0


In [14]:
# ── Config ────────────────────────────────────────────────────────────────────
A, B, C = 10.0, 6.0, 4.0        # ellipsoid semi-axes (dimensionless)

# UV mesh resolutions: (N_u, N_v) = (longitudes, latitude rings excl. poles)
# Two additional pole vertices are added per mesh to make it watertight.
RESOLUTIONS = [
    (50,   26,  'coarse'),
    (200, 101,  'medium'),
    (500, 251,  'fine'),
]

MESH_DIR = '/tmp/ellipsoid_mesh_test'
os.makedirs(MESH_DIR, exist_ok=True)

RANK2   = ['w020', 'w120', 'w220', 'w320', 'w102', 'w202']
SCALARS = ['w000', 'w100', 'w200', 'w300']

print(f'Ellipsoid semi-axes : A={A}, B={B}, C={C}')
print(f'Mesh directory      : {MESH_DIR}')
for Nu, Nv, label in RESOLUTIONS:
    n_verts = Nu * Nv + 2          # +2 pole vertices
    n_faces = 2 * Nu * (Nv - 1) + 2 * Nu  # quads + pole caps
    print(f'  {label:8s}: {Nu}×{Nv} grid + 2 poles  →  {n_verts:,} verts, {n_faces:,} faces')

Ellipsoid semi-axes : A=10.0, B=6.0, C=4.0
Mesh directory      : /tmp/ellipsoid_mesh_test
  coarse  : 50×26 grid + 2 poles  →  1,302 verts, 2,600 faces
  medium  : 200×101 grid + 2 poles  →  20,202 verts, 40,400 faces
  fine    : 500×251 grid + 2 poles  →  125,502 verts, 251,000 faces


In [15]:
# ── Analytical / numerical reference values ───────────────────────────────────
#
# Scalars
# -------
# w000 = (4/3)π A B C                          (exact)
# w100 = (1/3) ∫ dA                            (numerical, = surface_area / 3)
# w200 = (1/3) ∫ H dA   H = mean curvature    (numerical)
# w300 = (1/3) ∫ K dA = (1/3)·4π = 4π/3      (exact, Gauss–Bonnet, χ=2)
#
# Tensor betas
# ------------
# w020 : exact formula  W_ii = (4π/15) li³ lj lk
# others: numerical integration of the tensor over the parametric surface

def _analytical_scalars_and_betas(lx, ly, lz, N_u=600, N_v=300):
    """Return exact/numerical analytical values for all scalars and tensor betas."""
    u = np.linspace(0, 2*np.pi, N_u, endpoint=False)
    v = np.linspace(1e-5, np.pi - 1e-5, N_v)
    U, V = np.meshgrid(u, v, indexing='ij')

    X = lx * np.cos(U) * np.sin(V)
    Y = ly * np.sin(U) * np.sin(V)
    Z = lz * np.cos(V)

    # Unnormalised outward normal vector p = r_u × r_v / sin(V)
    Px = ly * lz * np.cos(U) * np.sin(V)
    Py = lx * lz * np.sin(U) * np.sin(V)
    Pz = lx * ly * np.cos(V)
    Pm = np.sqrt(Px**2 + Py**2 + Pz**2)       # |p|
    nx, ny, nz = Px/Pm, Py/Pm, Pz/Pm          # unit normal

    # Area element: dA = |r_u × r_v| du dv = sin(V) |p| du dv
    dA = np.sin(V) * Pm * (u[1]-u[0]) * (v[1]-v[0])

    # First fundamental form coefficients
    E   = lx**2*np.sin(U)**2*np.sin(V)**2 + ly**2*np.cos(U)**2*np.sin(V)**2
    F   = np.sin(U)*np.cos(U)*np.sin(V)*np.cos(V)*(ly**2 - lx**2)
    G   = lx**2*np.cos(U)**2*np.cos(V)**2 + ly**2*np.sin(U)**2*np.cos(V)**2 + lz**2*np.sin(V)**2
    EGF2 = E*G - F**2

    # Second fundamental form coefficients (for ellipsoid)
    L_sff = -lx * ly * lz * np.sin(V)**2 / Pm
    N_sff = -lx * ly * lz / Pm

    # Mean curvature H (positive for convex shape)
    H = -((E*N_sff + G*L_sff) / (2*EGF2))

    # Gaussian curvature K
    K = L_sff * N_sff / EGF2

    # ── Scalars ──────────────────────────────────────────────────────────────
    w000_ana = (4/3) * np.pi * lx * ly * lz          # exact
    w100_ana = np.sum(dA) / 3                          # S / 3
    w200_ana = np.sum(H * dA) / 3                      # ∫H dA / 3
    w300_ana = 4 * np.pi / 3                           # exact (Gauss–Bonnet)

    # ── Tensor betas ─────────────────────────────────────────────────────────
    def _beta(T):
        ev = np.sort(np.linalg.eigvalsh(T))
        return float(ev[0] / ev[2])

    def _tensor(ax, ay, az, wt):
        a = [ax.ravel(), ay.ravel(), az.ravel()]
        wr = wt.ravel()
        T = np.zeros((3, 3))
        for i in range(3):
            for j in range(3):
                T[i, j] = np.dot(a[i] * wr, a[j])
        return T

    # w020: exact formula  W_ii = (4π/15) li³ lj lk
    ev020 = np.sort([(4*np.pi/15)*li**3*lj*lk
                     for li, lj, lk in [(lx,ly,lz),(ly,lx,lz),(lz,lx,ly)]])
    beta_w020 = float(ev020[0] / ev020[2])

    betas = {
        'w020': beta_w020,
        'w120': _beta(_tensor(X, Y, Z, dA)),
        'w102': _beta(_tensor(nx, ny, nz, dA)),
        'w220': _beta(_tensor(X, Y, Z, H * dA)),
        'w320': _beta(_tensor(X, Y, Z, K * dA)),
        'w202': _beta(_tensor(nx, ny, nz, H * dA)),
    }

    scalars = {
        'w000': w000_ana,
        'w100': w100_ana,
        'w200': w200_ana,
        'w300': w300_ana,
    }
    return scalars, betas


print('Computing analytical reference values ...')
ANA_SCALARS, ANA_BETA = _analytical_scalars_and_betas(A, B, C)

print('\nAnalytical scalars:')
for sc, val in ANA_SCALARS.items():
    tag = '(exact)' if sc in ('w000', 'w300') else '(numerical)'
    print(f'  {sc}: {val:.6f}  {tag}')

print('\nAnalytical tensor betas:')
for name, val in ANA_BETA.items():
    tag = '(exact)' if name == 'w020' else '(numerical)'
    print(f'  {name}: {val:.6f}  {tag}')

Computing analytical reference values ...

Analytical scalars:
  w000: 1005.309649  (exact)
  w100: 179.697937  (numerical)
  w200: 29.061485  (numerical)
  w300: 4.188790  (exact)

Analytical tensor betas:
  w020: 0.160000  (exact)
  w120: 0.222136  (numerical)
  w102: 0.227565  (numerical)
  w220: 0.114710  (numerical)
  w320: 0.053368  (numerical)
  w202: 0.469470  (numerical)


In [16]:
# ── Build parametric UV meshes and save as .off ───────────────────────────────
#
# Vertex layout:
#   indices 0 … N_u*N_v-1  : ring vertices, row-major (u_i, v_j) → i*N_v + j
#   index   N_u*N_v         : north pole  (0, 0,  C)
#   index   N_u*N_v + 1     : south pole  (0, 0, -C)
#
# v is sampled strictly inside (0, π) to avoid degenerate ring vertices;
# the poles are added as single dedicated vertices and connected via fan
# triangles so the mesh is closed (watertight) → w000 is well-defined.
#
# Winding convention: outward-pointing normals (right-hand rule, CCW from outside).
# The parametric cross product r_u × r_v points INWARD for this UV parametrisation,
# so quads are wound (v00, v11, v10) / (v00, v01, v11) (reversed from naive order).
#
# OFF format:
#   OFF
#   <n_verts> <n_faces> 0
#   x y z           (one per vertex)
#   3 v0 v1 v2      (one per triangle)

def make_ellipsoid_off(lx, ly, lz, N_u, N_v, filepath):
    """Build a closed UV-triangulated ellipsoid and write it as an OFF file."""
    u = np.linspace(0, 2*np.pi, N_u, endpoint=False)      # N_u longitudes
    v = np.linspace(np.pi / (N_v + 1),                     # avoid exact poles
                    np.pi * N_v / (N_v + 1),
                    N_v)
    U, V = np.meshgrid(u, v, indexing='ij')               # shape (N_u, N_v)

    # Ring vertices
    ring_verts = np.column_stack([
        lx * np.cos(U).ravel() * np.sin(V).ravel(),
        ly * np.sin(U).ravel() * np.sin(V).ravel(),
        lz * np.cos(V).ravel(),
    ])  # shape (N_u*N_v, 3)

    # Pole vertices
    north_idx = N_u * N_v
    south_idx = N_u * N_v + 1
    pole_verts = np.array([[0.0, 0.0, lz],    # north pole
                           [0.0, 0.0, -lz]])  # south pole

    verts = np.vstack([ring_verts, pole_verts])

    faces = []

    # ── Interior quads (wound for outward normals) ──────────────────────────
    for i in range(N_u):
        i_next = (i + 1) % N_u
        for j in range(N_v - 1):
            v00 = i      * N_v + j
            v10 = i_next * N_v + j
            v01 = i      * N_v + (j + 1)
            v11 = i_next * N_v + (j + 1)
            # Reversed winding: r_u×r_v points inward, so flip to get outward
            faces.append((v00, v11, v10))
            faces.append((v00, v01, v11))

    # ── North pole cap ──────────────────────────────────────────────────────
    # Top ring: j=0, vertices at v = v[0]  (near north pole)
    # Viewed from outside (above), u increases CCW → wind (north, v_i, v_{i+1}) = CCW = outward
    for i in range(N_u):
        i_next = (i + 1) % N_u
        vi     = i      * N_v + 0
        vi_next = i_next * N_v + 0
        faces.append((north_idx, vi, vi_next))

    # ── South pole cap ──────────────────────────────────────────────────────
    # Bottom ring: j=N_v-1, vertices at v = v[-1]  (near south pole)
    # Viewed from outside (below), u increases CW → wind (south, v_{i+1}, v_i) = CCW = outward
    for i in range(N_u):
        i_next = (i + 1) % N_u
        vi      = i      * N_v + (N_v - 1)
        vi_next = i_next * N_v + (N_v - 1)
        faces.append((south_idx, vi_next, vi))

    faces = np.array(faces, dtype=np.int32)
    n_v, n_f = len(verts), len(faces)

    with open(filepath, 'w') as fh:
        fh.write('OFF\n')
        fh.write(f'{n_v} {n_f} 0\n')
        for x, y, z in verts:
            fh.write(f'{x:.10f} {y:.10f} {z:.10f}\n')
        for f0, f1, f2 in faces:
            fh.write(f'3 {f0} {f1} {f2}\n')

    return filepath, n_v, n_f


OFF_FILES = {}
print('Building ellipsoid meshes ...')
for Nu, Nv, label in RESOLUTIONS:
    path = os.path.join(MESH_DIR, f'ellipsoid_Nu{Nu}_Nv{Nv}.off')
    _, nv, nf = make_ellipsoid_off(A, B, C, Nu, Nv, path)
    OFF_FILES[label] = path
    print(f'  {label:8s}: {nv:,} verts, {nf:,} faces  →  {path}')

print('\nDone.  Files saved to', MESH_DIR)

Building ellipsoid meshes ...
  coarse  : 1,302 verts, 2,600 faces  →  /tmp/ellipsoid_mesh_test/ellipsoid_Nu50_Nv26.off
  medium  : 20,202 verts, 40,400 faces  →  /tmp/ellipsoid_mesh_test/ellipsoid_Nu200_Nv101.off
  fine    : 125,502 verts, 251,000 faces  →  /tmp/ellipsoid_mesh_test/ellipsoid_Nu500_Nv251.off

Done.  Files saved to /tmp/ellipsoid_mesh_test


In [17]:
# ── Run pykarambola on each mesh ──────────────────────────────────────────────
LONG_IDX = {'w020': 2, 'w120': 2, 'w220': 2, 'w320': 2, 'w102': 0, 'w202': 0}

MESH_RESULTS = {}
print('Running pykarambola.minkowski_tensors() on each mesh ...')
for Nu, Nv, label in RESOLUTIONS:
    path = OFF_FILES[label]
    tri  = pk.parse_off_file(path)
    r    = pk.minkowski_tensors(tri)

    scalars = {sc: float(r.get(sc, float('nan'))) for sc in SCALARS}

    betas = {}
    for name in RANK2:
        ev = r.get(f'{name}_eigvals')
        betas[name] = float(ev[0] / ev[2]) if ev is not None else float('nan')

    MESH_RESULTS[label] = {'scalars': scalars, 'betas': betas, 'n_faces': 2*Nu*(Nv-1)}
    print(f'  {label:8s}: done')

print('\nAll measurements complete.')

Running pykarambola.minkowski_tensors() on each mesh ...
  coarse  : done
  medium  : done
  fine    : done

All measurements complete.


In [18]:
# ── Comparison table: scalars ─────────────────────────────────────────────────
labels_ordered = [label for _, _, label in RESOLUTIONS]

col_w = 14
hdr   = f"{'scalar':<8}  {'analytical':>{col_w}}  "
hdr  += ''.join(f"{lb:>{col_w}}  " for lb in labels_ordered)
hdr  += ''.join(f"{'diff% '+lb:>{col_w}}  " for lb in labels_ordered)
sep   = '-' * len(hdr)

print('Scalars — pykarambola vs. analytical reference')
print('(diff% = 100 × |pykarambola − analytical| / |analytical|)')
print(sep)
print(hdr)
print(sep)

for sc in SCALARS:
    ana = ANA_SCALARS[sc]
    tag = '(exact)' if sc in ('w000', 'w300') else '(num.)'
    row = f"{sc+'  '+tag:<8}  {ana:>{col_w}.6f}  "
    diffs = []
    for lb in labels_ordered:
        val = MESH_RESULTS[lb]['scalars'][sc]
        row += f'{val:>{col_w}.6f}  '
        diffs.append(100 * abs(val - ana) / abs(ana) if abs(ana) > 1e-12 else float('nan'))
    for d in diffs:
        row += f'{d:>{col_w}.4f}  '
    print(row)

print(sep)

Scalars — pykarambola vs. analytical reference
(diff% = 100 × |pykarambola − analytical| / |analytical|)
--------------------------------------------------------------------------------------------------------------------------
scalar        analytical          coarse          medium            fine    diff% coarse    diff% medium      diff% fine  
--------------------------------------------------------------------------------------------------------------------------
w000  (exact)     1005.309649      999.276035     1004.905930     1005.244132          0.6002          0.0402          0.0065  
w100  (num.)      179.697937      179.117639      179.661784      179.694012          0.3229          0.0201          0.0022  
w200  (num.)       29.061485       29.028767       29.059478       29.061305          0.1126          0.0069          0.0006  
w300  (exact)        4.188790        4.188790        4.188790        4.188790          0.0000          0.0000          0.0000  
----------------

In [19]:
# ── Comparison table: tensor betas ───────────────────────────────────────────
hdr2  = f"{'tensor':<8}  {'analytical':>{col_w}}  "
hdr2 += ''.join(f"{lb:>{col_w}}  " for lb in labels_ordered)
hdr2 += ''.join(f"{'diff% '+lb:>{col_w}}  " for lb in labels_ordered)
sep2  = '-' * len(hdr2)

print('Tensor betas — pykarambola vs. analytical reference')
print('(diff% = 100 × |pykarambola β − analytical β| / |analytical β|)')
print(sep2)
print(hdr2)
print(sep2)

for name in RANK2:
    ana = ANA_BETA[name]
    tag = '(exact)' if name == 'w020' else '(num.)'
    row = f"{name+'  '+tag:<8}  {ana:>{col_w}.6f}  "
    diffs = []
    for lb in labels_ordered:
        val = MESH_RESULTS[lb]['betas'][name]
        row += f'{val:>{col_w}.6f}  '
        diffs.append(100 * abs(val - ana) / abs(ana) if abs(ana) > 1e-12 else float('nan'))
    for d in diffs:
        row += f'{d:>{col_w}.4f}  '
    print(row)

print(sep2)
print()
print('NOTE: if diff% is small here but large in the voxel notebook, the discrepancy')
print('is due to marching-cubes discretisation, not pykarambola\'s tensor operators.')
print('If diff% is similarly large here, the issue is in the operators themselves.')

Tensor betas — pykarambola vs. analytical reference
(diff% = 100 × |pykarambola β − analytical β| / |analytical β|)
--------------------------------------------------------------------------------------------------------------------------
tensor        analytical          coarse          medium            fine    diff% coarse    diff% medium      diff% fine  
--------------------------------------------------------------------------------------------------------------------------
w020  (exact)        0.160000        0.160421        0.160026        0.160004          0.2632          0.0164          0.0026  
w120  (num.)        0.222136        0.222354        0.222155        0.222145          0.0981          0.0086          0.0041  
w220  (num.)        0.114710        0.115151        0.114741        0.114718          0.3844          0.0267          0.0062  
w320  (num.)        0.053368        0.053715        0.053392        0.053372          0.6508          0.0458          0.0086  
w102  

In [20]:
# ── Print .off file paths for karambola (C++) testing ─────────────────────────
print('OFF files ready for karambola:')
for Nu, Nv, label in RESOLUTIONS:
    path = OFF_FILES[label]
    nf   = MESH_RESULTS[label]['n_faces']
    print(f'  {label:8s} ({nf:,} faces): {path}')

OFF files ready for karambola:
  coarse   (2,500 faces): /tmp/ellipsoid_mesh_test/ellipsoid_Nu50_Nv26.off
  medium   (40,000 faces): /tmp/ellipsoid_mesh_test/ellipsoid_Nu200_Nv101.off
  fine     (250,000 faces): /tmp/ellipsoid_mesh_test/ellipsoid_Nu500_Nv251.off


---
## Voxel-derived meshes vs. parametric UV meshes — scalar comparison

Recreate the voxel masks exactly as `orientation_bias_archive.ipynb` does, then extract
the mesh using the same pipeline that `minkowski_tensors_from_label_image()` uses internally:

1. `np.pad` (1-voxel border, same as `pad=True` default)
2. `skimage.marching_cubes(level=0.5, gradient_direction='ascent')`
3. Subtract the pad offset to restore physical coordinates
4. `_ensure_outward_normals` — the exact outward-normal fix applied internally

The resulting meshes are saved as `.off` and measured with `pk.minkowski_tensors()`.
Scalars are then compared against the parametric UV mesh results and the analytical reference.

In [21]:
# ── Voxel-mesh extraction — exact pipeline from minkowski_tensors_from_label_image ──
#
# Voxel sizes: same three denominators as in orientation_bias_archive.ipynb
# (A/3 = coarse, A/20 = medium, A/60 = fine), angle fixed at 0°.

from skimage.measure import marching_cubes
from pykarambola.api import _ensure_outward_normals

VOX_DENOMS  = [3, 20, 60]
VOX_LABELS  = [f'A/{n}' for n in VOX_DENOMS]
VOXEL_SIZES = [A / n for n in VOX_DENOMS]


def make_ellipsoid_mask(lx, ly, lz, voxel_size, angle_deg=0, pad=4):
    """Binary voxel mask of a triaxial ellipsoid (copied from orientation_bias_archive)."""
    theta = np.deg2rad(angle_deg)
    n0 = int(np.ceil(max(lx, ly) / voxel_size)) + pad
    n1 = int(np.ceil(max(lx, ly) / voxel_size)) + pad
    n2 = int(np.ceil(lz           / voxel_size)) + pad
    X0, X1, X2 = np.meshgrid(
        np.arange(-n0, n0+1) * voxel_size,
        np.arange(-n1, n1+1) * voxel_size,
        np.arange(-n2, n2+1) * voxel_size,
        indexing='ij',
    )
    X0r =  X0 * np.cos(theta) + X1 * np.sin(theta)
    X1r = -X0 * np.sin(theta) + X1 * np.cos(theta)
    return ((X0r/lx)**2 + (X1r/ly)**2 + (X2/lz)**2 <= 1.0).astype(np.uint8)


def voxel_mask_to_mesh(mask, voxel_size):
    """Extract mesh from binary mask using the same pipeline as minkowski_tensors_from_label_image.

    Steps (mirroring pykarambola/api.py):
      1. Pad with a 1-voxel border (pad=True default)
      2. marching_cubes(level=0.5, spacing=voxel_size, gradient_direction='ascent')
      3. Subtract pad offset to restore physical coordinates
      4. _ensure_outward_normals
    """
    spacing    = (voxel_size, voxel_size, voxel_size)
    padded     = np.pad(mask, pad_width=1, mode='constant', constant_values=0).astype(np.float64)
    pad_offset = np.array(spacing, dtype=np.float64)

    verts, faces, _, _ = marching_cubes(
        padded, level=0.5, spacing=spacing, gradient_direction='ascent'
    )
    verts  = verts - pad_offset
    faces  = _ensure_outward_normals(verts, faces)
    return verts, faces


def write_off(verts, faces, filepath):
    with open(filepath, 'w') as fh:
        fh.write('OFF\n')
        fh.write(f'{len(verts)} {len(faces)} 0\n')
        for x, y, z in verts:
            fh.write(f'{x:.10f} {y:.10f} {z:.10f}\n')
        for f0, f1, f2 in faces:
            fh.write(f'3 {f0} {f1} {f2}\n')


VOX_OFF_FILES = {}
VOX_MASKS     = {}   # raw binary masks (needed for skimage scalar computation)
VOX_MESHES    = {}   # (verts, faces) tuples (needed for mesh_surface_area)

print('Extracting voxel-derived meshes ...')
for denom, label, vox in zip(VOX_DENOMS, VOX_LABELS, VOXEL_SIZES):
    mask         = make_ellipsoid_mask(A, B, C, vox, angle_deg=0)
    verts, faces = voxel_mask_to_mesh(mask, vox)
    path         = os.path.join(MESH_DIR, f'ellipsoid_vox_A{denom}.off')
    write_off(verts, faces, path)

    VOX_MASKS[label]     = mask
    VOX_MESHES[label]    = (verts, faces)
    VOX_OFF_FILES[label] = path
    print(f'  {label}: {len(verts):,} verts, {len(faces):,} faces  →  {path}')

print('\nDone.')

Extracting voxel-derived meshes ...
  A/3: 70 verts, 136 faces  →  /tmp/ellipsoid_mesh_test/ellipsoid_vox_A3.off
  A/20: 3,086 verts, 6,168 faces  →  /tmp/ellipsoid_mesh_test/ellipsoid_vox_A20.off
  A/60: 27,950 verts, 55,896 faces  →  /tmp/ellipsoid_mesh_test/ellipsoid_vox_A60.off

Done.


In [22]:
# ── Measure scalars on voxel-derived meshes ───────────────────────────────────
VOX_SCALAR_RESULTS = {}
print('Running pk.minkowski_tensors() on voxel-derived meshes ...')
for label in VOX_LABELS:
    tri = pk.parse_off_file(VOX_OFF_FILES[label])
    r   = pk.minkowski_tensors(tri)
    VOX_SCALAR_RESULTS[label] = {sc: float(r.get(sc, float('nan'))) for sc in SCALARS}
    print(f'  {label}: done')

print('\nDone.')

Running pk.minkowski_tensors() on voxel-derived meshes ...
  A/3: done
  A/20: done
  A/60: done

Done.


In [23]:
# ── Skimage-based volume directly from voxel representation ───────────────────
#
# w000 = np.sum(mask) * vox³   (voxel counting — no mesh required)

VOX_SKIMAGE_RESULTS = {}
print('Computing skimage-based volume from voxel masks ...')
for denom, label, vox in zip(VOX_DENOMS, VOX_LABELS, VOXEL_SIZES):
    mask = VOX_MASKS[label]
    w000 = float(np.sum(mask)) * vox**3
    VOX_SKIMAGE_RESULTS[label] = {'w000': w000, 'w100': float('nan'),
                                   'w200': float('nan'), 'w300': float('nan')}
    print(f'  {label}:  w000 = {w000:.6f}')

print('\nDone.')

Computing skimage-based volume from voxel masks ...
  A/3:  w000 = 851.851852
  A/20:  w000 = 1002.625000
  A/60:  w000 = 1004.282407

Done.


In [24]:
# ── Side-by-side scalar comparison: skimage / pykarambola / analytical ─────────
#
# For each scalar, one table covers all three voxel resolutions.
# Columns: skimage (A/3, A/20, A/60) | pykarambola-vox (A/3, A/20, A/60) | analytical
# Diff% is vs analytical for every value.  w200 skimage = n/a.

UV_LABELS = [label for _, _, label in RESOLUTIONS]

CW = 11

def _fmt(v):
    return f'{"n/a":>{CW}}' if (v != v) else f'{v:{CW}.5f}'

def _pct(v, ana):
    if v != v or abs(ana) < 1e-12:
        return f'{"n/a":>{CW}}'
    return f'{100*abs(v-ana)/abs(ana):{CW}.3f}'

print('Scalar comparison: skimage (voxels) vs pykarambola (voxel mesh) vs analytical')
print('(diff% = 100 × |value − analytical| / |analytical|)')
print()

for sc in SCALARS:
    ana = ANA_SCALARS[sc]
    tag = '(exact)' if sc in ('w000', 'w300') else '(num.)'

    sk_vals = [VOX_SKIMAGE_RESULTS[lb][sc] for lb in VOX_LABELS]
    pk_vals = [VOX_SCALAR_RESULTS[lb][sc]  for lb in VOX_LABELS]

    vl_str = ''.join(f'{lb:{CW}}' for lb in VOX_LABELS)
    hdr = (f'{"method":<14}  {vl_str}   '
           f'{"Δ% "+VOX_LABELS[0]:{CW}}  {"Δ% "+VOX_LABELS[1]:{CW}}  {"Δ% "+VOX_LABELS[2]:{CW}}')
    sep = '-' * len(hdr)

    print(f'{sc}  {tag}  analytical = {ana:.6f}')
    print(sep)
    print(hdr)
    print(sep)

    for method, vals in [('skimage', sk_vals), ('pykarambola', pk_vals)]:
        val_str  = ''.join(_fmt(v)        for v in vals)
        diff_str = '  '.join(_pct(v, ana) for v in vals)
        print(f'{method:<14}  {val_str}   {diff_str}')

    ana_str = ''.join(f'{ana:{CW}.5f}' for _ in VOX_LABELS)
    print(f'{"analytical":<14}  {ana_str}')
    print(sep)
    print()

Scalar comparison: skimage (voxels) vs pykarambola (voxel mesh) vs analytical
(diff% = 100 × |value − analytical| / |analytical|)

w000  (exact)  analytical = 1005.309649
-----------------------------------------------------------------------------------------
method          A/3        A/20       A/60          Δ% A/3       Δ% A/20      Δ% A/60    
-----------------------------------------------------------------------------------------
skimage           851.85185 1002.62500 1004.28241        15.265        0.267        0.102
pykarambola       685.18519  999.02083 1003.88812        31.843        0.626        0.141
analytical       1005.30965 1005.30965 1005.30965
-----------------------------------------------------------------------------------------

w100  (num.)  analytical = 179.697937
-----------------------------------------------------------------------------------------
method          A/3        A/20       A/60          Δ% A/3       Δ% A/20      Δ% A/60    
--------------------